# Session 3 · Part 2 — Evaluate spatial coherence

**Goal:** test whether DGAT preserves local tissue structure. Moran's I compares nearby values under a
spatial-neighborhood definition: positive values indicate similar neighbors, values near zero indicate
spatial randomness, and negative values indicate local dissimilarity. This complements—not replaces—
pointwise correlation.


In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
for candidate in (current, *current.parents):
    if (candidate / "src" / "dgat_tutorial").is_dir():
        tutorial_root = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter inside the hands-on_tutorial directory.")

sys.path.insert(0, str(tutorial_root / "src"))

from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint

paths = tutorial_paths(tutorial_root)
print(f"Tutorial root: {paths.root}")


## 1. Load aligned observations, predictions, and coordinates


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from dgat_tutorial.checkpoints import preferred_prediction_path
from dgat_tutorial.data import load_tutorial_data
from dgat_tutorial.dgat import load_prediction_table
from dgat_tutorial.evaluation import morans_i

dataset = load_tutorial_data(paths.raw_data)
predicted = load_prediction_table(str(preferred_prediction_path(paths)))
common_spots = dataset.spots.index.intersection(dataset.proteins.index).intersection(predicted.index)
common_proteins = dataset.proteins.columns.intersection(predicted.columns)
if common_spots.empty or common_proteins.empty:
    raise ValueError("Observed and predicted data do not share both spot IDs and protein names.")
spots = dataset.spots.loc[common_spots]
observed = dataset.proteins.loc[common_spots, common_proteins]
predicted = predicted.loc[common_spots, common_proteins]


## 2. Calculate observed and predicted Moran's I with the same neighborhood rule


In [ ]:
moran_table = pd.DataFrame([
    {"protein": protein,
     "observed_morans_i": morans_i(observed[protein], spots),
     "predicted_morans_i": morans_i(predicted[protein], spots)}
    for protein in common_proteins
])
moran_table["difference"] = moran_table["predicted_morans_i"] - moran_table["observed_morans_i"]
moran_table.sort_values("difference")


### Figure 12 — Observed versus predicted spatial coherence


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(data=moran_table, x="observed_morans_i", y="predicted_morans_i", hue="protein", s=70, ax=ax)
limits = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
ax.plot(limits, limits, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Spatial coherence: points on the line are preserved")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7, frameon=False)
moran_scatter_path = paths.figures / "session03_morans_i.png"
fig.tight_layout(); fig.savefig(moran_scatter_path, dpi=160, bbox_inches="tight"); plt.show()


### Figure 13 — Which proteins are over-smoothed or under-smoothed?


In [ ]:
ordered = moran_table.sort_values("difference")
fig, ax = plt.subplots(figsize=(7, max(4, 0.24 * len(ordered))))
colors = ["#b65f3c" if value < 0 else "#4c78a8" for value in ordered["difference"]]
ax.barh(ordered["protein"], ordered["difference"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set(xlabel="predicted Moran's I − observed Moran's I", title="Spatial smoothing bias by protein")
difference_path = paths.figures / "session03_morans_i_difference.png"
fig.tight_layout(); fig.savefig(difference_path, dpi=160, bbox_inches="tight"); plt.show()


Positive differences can indicate over-smoothing; negative differences can indicate lost spatial signal.
Interpretation depends on the graph radius and tissue geometry, so report the neighborhood rule and compare
proteins under the same rule.


In [ ]:
table_path = paths.results / "session03_morans_i.csv"
moran_table.to_csv(table_path, index=False)
manifest = write_checkpoint(
    "3.2", [table_path, moran_scatter_path, difference_path],
    summary={"proteins_evaluated": len(moran_table)}, start=paths.root,
)
print(f"Checkpoint written: {manifest}")


## Check

Find one protein whose pointwise correlation and spatial-coherence result agree, and one where they tell
different stories. The disagreement is often the most useful result.
